In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/us_gun_policy_supervised_v3.csv"
)

treino = df[df["year"] <= 2022].copy()
teste = df[df["year"] == 2023].copy()

print("Treino:", treino.shape)
print("Teste:", teste.shape)

print("\nAnos de treinamento:", sorted(treino["year"].unique()))
print("Anos de teste:", sorted(teste["year"].unique()))

Treino: (200, 13)
Teste: (50, 13)

Anos de treinamento: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Anos de teste: [np.int64(2023)]


In [2]:
variaveis_a = [
    "homicide_rate",
    "unemployment_rate",
    "income_per_capita"
]

variaveis_legislacao = [
    "ccw_shall_issue",
    "ccw_permitless",
    "ccw_may_issue",
    "ccw_prohibited",
    "background_handgun",
    "background_longgun"
]

variaveis_b = variaveis_a + variaveis_legislacao

alvo = "homicide_rate_next_year"

print("Modelo A:", variaveis_a)
print("\nModelo B:", variaveis_b)
print("\nVariável-alvo:", alvo)

print("\nIndicadores legislativos por ano:")
display(
    treino.groupby("year")[variaveis_legislacao].sum()
)

Modelo A: ['homicide_rate', 'unemployment_rate', 'income_per_capita']

Modelo B: ['homicide_rate', 'unemployment_rate', 'income_per_capita', 'ccw_shall_issue', 'ccw_permitless', 'ccw_may_issue', 'ccw_prohibited', 'background_handgun', 'background_longgun']

Variável-alvo: homicide_rate_next_year

Indicadores legislativos por ano:


,ccw_shall_issue,ccw_permitless,ccw_may_issue,ccw_prohibited,background_handgun,background_longgun
year,,,,,,
2019,27,15,8,0,18,15
2020,27,15,8,0,19,16
2021,21,21,8,0,19,16
2022,24,24,2,0,18,15


In [3]:
colunas_ccw = [
    "ccw_shall_issue",
    "ccw_permitless",
    "ccw_may_issue",
    "ccw_prohibited"
]

df["total_ccw"] = df[colunas_ccw].sum(axis=1)

print("Distribuição das modalidades por estado-ano:")
display(df["total_ccw"].value_counts().sort_index())

print("\nEstados com mais de uma modalidade:")
display(
    df.loc[
        df["total_ccw"] > 1,
        ["state", "year"] + colunas_ccw
    ]
)

Distribuição das modalidades por estado-ano:


total_ccw
1    250
Name: count, dtype: int64


Estados com mais de uma modalidade:


,state,year,ccw_shall_issue,ccw_permitless,ccw_may_issue,ccw_prohibited


In [4]:
ccw = df.sort_values(["state", "year"]).copy()

ccw["modalidade"] = ccw[colunas_ccw].idxmax(axis=1)

ccw["modalidade_anterior"] = (
    ccw.groupby("state")["modalidade"].shift(1)
)

mudancas = ccw[
    ccw["modalidade_anterior"].notna()
    & (ccw["modalidade"] != ccw["modalidade_anterior"])
]

display(
    mudancas[
        ["state", "year", "modalidade_anterior", "modalidade"]
    ].sort_values(["year", "state"])
)

print("Total de mudanças:", len(mudancas))

,state,year,modalidade_anterior,modalidade
17,Arkansas,2021,ccw_shall_issue,ccw_permitless
72,Iowa,2021,ccw_shall_issue,ccw_permitless
127,Montana,2021,ccw_shall_issue,ccw_permitless
207,Tennessee,2021,ccw_shall_issue,ccw_permitless
212,Texas,2021,ccw_shall_issue,ccw_permitless
217,Utah,2021,ccw_shall_issue,ccw_permitless
23,California,2022,ccw_may_issue,ccw_shall_issue
48,Georgia,2022,ccw_shall_issue,ccw_permitless
53,Hawaii,2022,ccw_may_issue,ccw_shall_issue
68,Indiana,2022,ccw_shall_issue,ccw_permitless


Total de mudanças: 18


In [5]:
import pandas as pd

arquivo_rand = (
    "../reports/attachments/"
    "TL-A243-2-v4 State Firearm Law Database 6.0.xlsx"
)

rand = pd.read_excel(
    arquivo_rand,
    sheet_name="Database"
)

print(rand.columns.tolist())

['Law ID', 'State', 'State Postal Abbreviation', 'FIPS Code', 'Law Class (num)', 'Law Class', 'Law Class Subtype', 'Handguns or Long Guns', 'Effect', 'Type of Change', 'Effective Date Note', 'Effective Date Year', 'Effective Date Month', 'Effective Date Day', 'Statutory Citation', 'Content', 'Supersession Date Year', 'Supersession Date Month', 'Supersession Date Day', 'Age for Minimum Age Laws', 'Length of Waiting Period (days, handguns)', 'Additional Context and Notes', 'Discrepancies']


In [6]:
estados_bruen = [
    "California",
    "Hawaii",
    "Maryland",
    "Massachusetts",
    "New Jersey",
    "New York"
]

colunas_inspecao = [
    "Law ID",
    "State",
    "Law Class Subtype",
    "Effect",
    "Type of Change",
    "Effective Date Note",
    "Effective Date Year",
    "Effective Date Month",
    "Effective Date Day",
    "Supersession Date Year",
    "Additional Context and Notes"
]

rand_bruen = rand[
    rand["State"].isin(estados_bruen)
    & rand["Law Class"].astype(str).str.contains(
        "carrying a concealed weapon",
        case=False,
        na=False
    )
    & rand["Effective Date Year"].between(2021, 2023)
]

print("Registros encontrados:", len(rand_bruen))

display(rand_bruen[colunas_inspecao])

Registros encontrados: 7


,Law ID,State,Law Class Subtype,Effect,Type of Change,Effective Date Note,Effective Date Year,Effective Date Month,Effective Date Day,Supersession Date Year,Additional Context and Notes
178,CA1107,California,shall issue,Permissive,Modify,NaN,2022,6,23,NaN,NaN
522,HI1084,Hawaii,shall issue,Permissive,Modify,NaN,2022,6,23,NaN,The Supreme court decision in Bruen determined...
823,MA1068,Massachusetts,shall issue,Permissive,Modify,NaN,2022,6,23,NaN,The Supreme court decision in Bruen determined...
872,MD1065,Maryland,shall issue,Permissive,Modify,NaN,2022,6,23,NaN,The Supreme court decision in Bruen determined...
1236,NJ1072,New Jersey,shall issue,Permissive,Modify,NaN,2022,6,23,NaN,The Supreme court decision in Bruen determined...
1248,NJ1085,New Jersey,shall issue,Permissive,Modify,NaN,2022,12,22,NaN,NaN
1350,NY1069,New York,shall issue,Permissive,Modify,NaN,2022,6,23,NaN,The Supreme court decision in Bruen determined...


In [7]:
anteriores = rand[
    rand["State"].isin(estados_bruen)
    & rand["Law Class"].astype(str).str.contains(
        "carrying a concealed weapon",
        case=False,
        na=False
    )
    & rand["Law Class Subtype"].astype(str).str.contains(
        "may issue",
        case=False,
        na=False
    )
]

display(
    anteriores[
        [
            "Law ID",
            "State",
            "Law Class Subtype",
            "Effective Date Year",
            "Supersession Date Year",
            "Supersession Date Month",
            "Supersession Date Day"
        ]
    ]
)

,Law ID,State,Law Class Subtype,Effective Date Year,Supersession Date Year,Supersession Date Month,Supersession Date Day
112,CA1007,California,may issue,1953,2022.0,6.0,23.0
470,HI1005,Hawaii,may issue,1961,2022.0,6.0,23.0
781,MA1004,Massachusetts,may issue,1906,2022.0,6.0,23.0
840,MD1005,Maryland,may issue,1972,2022.0,6.0,23.0
1194,NJ1006,New Jersey,may issue,1924,2022.0,6.0,23.0
1315,NY1008,New York,may issue,1973,2022.0,6.0,23.0


In [8]:
estados_permitless = [
    "Arkansas", "Iowa", "Montana", "Tennessee",
    "Texas", "Utah", "Georgia", "Indiana",
    "Ohio", "Alabama", "Florida", "Nebraska"
]

registros_permitless = rand[
    rand["State"].isin(estados_permitless)
    & rand["Law Class"].astype(str).str.contains(
        "carrying a concealed weapon",
        case=False,
        na=False
    )
    & rand["Law Class Subtype"].astype(str).str.contains(
        "permitless",
        case=False,
        na=False
    )
    & rand["Effective Date Year"].between(2020, 2023)
]

display(
    registros_permitless[
        [
            "Law ID",
            "State",
            "Law Class Subtype",
            "Type of Change",
            "Effective Date Year",
            "Effective Date Month",
            "Effective Date Day"
        ]
    ].sort_values(["Effective Date Year", "State"])
)

print("Registros encontrados:", len(registros_permitless))

,Law ID,State,Law Class Subtype,Type of Change,Effective Date Year,Effective Date Month,Effective Date Day


Registros encontrados: 0


In [9]:
ccw_rand = rand[
    rand["Law Class"].astype(str).str.contains(
        "carrying a concealed weapon",
        case=False,
        na=False
    )
].copy()

print("Registros CCW:", len(ccw_rand))

print("\nSubcategorias disponíveis:")
display(ccw_rand["Law Class Subtype"].value_counts(dropna=False))

print("\nEfeitos disponíveis:")
display(ccw_rand["Effect"].value_counts(dropna=False))

Registros CCW: 142

Subcategorias disponíveis:


Law Class Subtype
shall issue                          56
may issue                            35
shall issue (permit not required)    29
prohibited                           22
Name: count, dtype: int64


Efeitos disponíveis:


Effect
Permissive     117
Restrictive     25
Name: count, dtype: int64

In [10]:
variaveis_a = [
    "homicide_rate",
    "unemployment_rate",
    "income_per_capita"
]

alvo = "homicide_rate_next_year"

X_train_a = treino[variaveis_a]
X_test_a = teste[variaveis_a]

y_train = treino[alvo]
y_test = teste[alvo]

print("X_train:", X_train_a.shape)
print("X_test:", X_test_a.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

display(X_train_a.head())

X_train: (200, 3)
X_test: (50, 3)
y_train: (200,)
y_test: (50,)


,homicide_rate,unemployment_rate,income_per_capita
0,12.0,3.2,42996.0
1,13.0,6.4,45874.0
2,14.8,3.4,50651.0
3,13.9,2.5,51706.0
5,10.7,5.6,60483.0


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

modelo_a = Pipeline([
    ("scaler", StandardScaler()),
    ("regressao", LinearRegression())
])

modelo_a.fit(X_train_a, y_train)

print("Modelo A treinado com sucesso!")

Modelo A treinado com sucesso!


In [13]:
# Prever as taxas de homicídios de 2024
y_pred_a = modelo_a.predict(X_test_a)

# Comparar previsões com os valores reais
resultados_a = teste[
    ["state", "year", "homicide_rate", "homicide_rate_next_year"]
].copy()

resultados_a["previsao_modelo_a"] = y_pred_a

resultados_a = resultados_a.rename(
    columns={
        "homicide_rate": "taxa_2023",
        "homicide_rate_next_year": "taxa_real_2024"
    }
)

display(resultados_a.head(10).round(2))

,state,year,taxa_2023,taxa_real_2024,previsao_modelo_a
4,Alabama,2023,14.0,12.6,13.71
9,Alaska,2023,8.3,7.6,7.58
14,Arizona,2023,7.1,6.3,7.02
19,Arkansas,2023,10.6,8.1,10.55
24,California,2023,5.1,4.5,4.18
29,Colorado,2023,6.3,5.2,5.30
34,Connecticut,2023,4.2,2.9,2.68
39,Delaware,2023,5.3,6.5,5.36
44,Florida,2023,6.0,5.2,5.51
49,Georgia,2023,9.7,9.1,9.53


In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae_a = mean_absolute_error(y_test, y_pred_a)

rmse_a = np.sqrt(
    mean_squared_error(y_test, y_pred_a)
)

print(f"MAE Modelo A: {mae_a:.3f}")
print(f"RMSE Modelo A: {rmse_a:.3f}")

MAE Modelo A: 0.677
RMSE Modelo A: 0.953


In [15]:
# Previsão ingênua: repetir a taxa do ano anterior
y_pred_ingenuo = teste["homicide_rate"]

mae_ingenuo = mean_absolute_error(y_test, y_pred_ingenuo)

rmse_ingenuo = np.sqrt(
    mean_squared_error(y_test, y_pred_ingenuo)
)

print(f"MAE Ingênuo: {mae_ingenuo:.3f}")
print(f"RMSE Ingênuo: {rmse_ingenuo:.3f}")

MAE Ingênuo: 0.838
RMSE Ingênuo: 1.072


In [16]:
variaveis_b = [
    "homicide_rate",
    "unemployment_rate",
    "income_per_capita",
    "ccw_permitless",
    "ccw_may_issue",
    "background_handgun",
    "background_longgun"
]

X_train_b = treino[variaveis_b]
X_test_b = teste[variaveis_b]

print("Treinamento B:", X_train_b.shape)
print("Teste B:", X_test_b.shape)

display(X_train_b.head())

Treinamento B: (200, 7)
Teste B: (50, 7)


,homicide_rate,unemployment_rate,income_per_capita,ccw_permitless,ccw_may_issue,background_handgun,background_longgun
0,12.0,3.2,42996.0,0,0,0,0
1,13.0,6.4,45874.0,0,0,0,0
2,14.8,3.4,50651.0,0,0,0,0
3,13.9,2.5,51706.0,0,0,0,0
5,10.7,5.6,60483.0,1,0,0,0


In [17]:
modelo_b = Pipeline([
    ("scaler", StandardScaler()),
    ("regressao", LinearRegression())
])

modelo_b.fit(X_train_b, y_train)

print("Modelo B treinado com sucesso!")

Modelo B treinado com sucesso!


In [18]:
y_pred_b = modelo_b.predict(X_test_b)

mae_b = mean_absolute_error(y_test, y_pred_b)

rmse_b = np.sqrt(
    mean_squared_error(y_test, y_pred_b)
)

print(f"MAE Modelo A: {mae_a:.3f}")
print(f"MAE Modelo B: {mae_b:.3f}")

print()

print(f"RMSE Modelo A: {rmse_a:.3f}")
print(f"RMSE Modelo B: {rmse_b:.3f}")

MAE Modelo A: 0.677
MAE Modelo B: 0.664

RMSE Modelo A: 0.953
RMSE Modelo B: 0.988


In [19]:
comparacao = teste[["state"]].copy()

comparacao["real_2024"] = y_test
comparacao["previsao_a"] = y_pred_a
comparacao["previsao_b"] = y_pred_b

comparacao["erro_a"] = abs(
    comparacao["real_2024"] - comparacao["previsao_a"]
)

comparacao["erro_b"] = abs(
    comparacao["real_2024"] - comparacao["previsao_b"]
)

comparacao["diferenca_erro"] = (
    comparacao["erro_b"] - comparacao["erro_a"]
)

display(
    comparacao.sort_values(
        "diferenca_erro", ascending=False
    ).head(10).round(3)
)

,state,real_2024,previsao_a,previsao_b,erro_a,erro_b,diferenca_erro
249,Wyoming,4.6,2.155,1.607,2.445,2.993,0.548
9,Alaska,7.6,7.582,7.073,0.018,0.527,0.508
154,New Mexico,12.7,13.462,13.926,0.762,1.226,0.464
204,South Dakota,6.0,3.638,3.244,2.362,2.756,0.394
184,Oregon,4.0,4.351,4.738,0.351,0.738,0.387
189,Pennsylvania,5.3,6.703,7.060,1.403,1.760,0.357
229,Virginia,5.5,5.876,6.211,0.376,0.711,0.335
139,Nevada,6.9,7.759,8.074,0.859,1.174,0.315
194,Rhode Island,1.8,2.678,2.950,0.878,1.150,0.272
64,Illinois,8.4,8.514,8.786,0.114,0.386,0.272


In [20]:
# Estados em que o Modelo B apresentou as maiores melhorias
melhorias_b = comparacao.sort_values(
    "diferenca_erro",
    ascending=True
)

display(melhorias_b.head(10).round(3))

print("Estados em que B melhorou:",
      (comparacao["diferenca_erro"] < 0).sum())

print("Estados em que B piorou:",
      (comparacao["diferenca_erro"] > 0).sum())

print("Estados com empate:",
      (comparacao["diferenca_erro"].abs() < 1e-10).sum())

,state,real_2024,previsao_a,previsao_b,erro_a,erro_b,diferenca_erro
214,Texas,6.1,6.756,6.318,0.656,0.218,-0.437
209,Tennessee,9.8,10.454,10.074,0.654,0.274,-0.380
14,Arizona,6.3,7.015,6.641,0.715,0.341,-0.374
124,Missouri,9.0,9.508,9.135,0.508,0.135,-0.372
174,Ohio,6.3,7.378,7.019,1.078,0.719,-0.359
79,Kansas,5.3,6.242,5.900,0.942,0.600,-0.342
69,Indiana,7.0,7.584,7.243,0.584,0.243,-0.342
49,Georgia,9.1,9.531,9.194,0.431,0.094,-0.337
94,Maine,2.4,3.488,3.154,1.088,0.754,-0.334
84,Kentucky,6.9,7.392,7.084,0.492,0.184,-0.308


Estados em que B melhorou: 24
Estados em que B piorou: 26
Estados com empate: 0


In [21]:
rodadas = [
    {"treino_ate": 2020, "teste_em": 2021},
    {"treino_ate": 2021, "teste_em": 2022},
    {"treino_ate": 2022, "teste_em": 2023}
]

for rodada in rodadas:
    treino_temp = df[df["year"] <= rodada["treino_ate"]]
    teste_temp = df[df["year"] == rodada["teste_em"]]

    print(
        f"Treino até {rodada['treino_ate']}: "
        f"{len(treino_temp)} registros | "
        f"Teste {rodada['teste_em']}: "
        f"{len(teste_temp)} registros"
    )

Treino até 2020: 100 registros | Teste 2021: 50 registros
Treino até 2021: 150 registros | Teste 2022: 50 registros
Treino até 2022: 200 registros | Teste 2023: 50 registros


In [22]:
resultados_temporais = []

for rodada in rodadas:
    treino_temp = df[df["year"] <= rodada["treino_ate"]]
    teste_temp = df[df["year"] == rodada["teste_em"]]

    y_treino = treino_temp[alvo]
    y_teste = teste_temp[alvo]

    for nome, variaveis in [
        ("Modelo A", variaveis_a),
        ("Modelo B", variaveis_b)
    ]:
        modelo = Pipeline([
            ("scaler", StandardScaler()),
            ("regressao", LinearRegression())
        ])

        modelo.fit(treino_temp[variaveis], y_treino)

        previsoes = modelo.predict(teste_temp[variaveis])

        resultados_temporais.append({
            "ano_previsto": rodada["teste_em"] + 1,
            "modelo": nome,
            "MAE": mean_absolute_error(y_teste, previsoes),
            "RMSE": np.sqrt(
                mean_squared_error(y_teste, previsoes)
            )
        })

resultado_temporal = pd.DataFrame(resultados_temporais)

display(resultado_temporal.round(3))

,ano_previsto,modelo,MAE,RMSE
0,2022,Modelo A,1.330,1.763
1,2022,Modelo B,1.380,1.822
2,2023,Modelo A,0.826,1.001
3,2023,Modelo B,0.832,0.979
4,2024,Modelo A,0.677,0.953
5,2024,Modelo B,0.664,0.988


In [23]:
resumo_temporal = (
    resultado_temporal
    .groupby("modelo")[["MAE", "RMSE"]]
    .mean()
    .round(3)
)

display(resumo_temporal)

,MAE,RMSE
modelo,,
Modelo A,0.944,1.239
Modelo B,0.959,1.263


In [24]:
resultados_ingenuos = []

for rodada in rodadas:
    teste_temp = df[df["year"] == rodada["teste_em"]]

    valores_reais = teste_temp["homicide_rate_next_year"]
    previsao_ingenua = teste_temp["homicide_rate"]

    resultados_ingenuos.append({
        "ano_previsto": rodada["teste_em"] + 1,
        "modelo": "Ingênuo",
        "MAE": mean_absolute_error(
            valores_reais, previsao_ingenua
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                valores_reais, previsao_ingenua
            )
        )
    })

resultado_completo = pd.concat(
    [
        resultado_temporal,
        pd.DataFrame(resultados_ingenuos)
    ],
    ignore_index=True
)

display(
    resultado_completo
    .sort_values(["ano_previsto", "modelo"])
    .round(3)
)

,ano_previsto,modelo,MAE,RMSE
6,2022,Ingênuo,0.808,1.126
0,2022,Modelo A,1.330,1.763
1,2022,Modelo B,1.380,1.822
7,2023,Ingênuo,0.710,0.884
2,2023,Modelo A,0.826,1.001
3,2023,Modelo B,0.832,0.979
8,2024,Ingênuo,0.838,1.072
4,2024,Modelo A,0.677,0.953
5,2024,Modelo B,0.664,0.988


In [25]:
resumo_final = (
    resultado_completo
    .groupby("modelo")[["MAE", "RMSE"]]
    .mean()
    .sort_values("MAE")
    .round(3)
)

display(resumo_final)

,MAE,RMSE
modelo,,
Ingênuo,0.785,1.027
Modelo A,0.944,1.239
Modelo B,0.959,1.263


In [26]:
from pathlib import Path

pasta = Path("../reports/results")
pasta.mkdir(parents=True, exist_ok=True)

resultado_completo.to_csv(
    pasta / "temporal_validation.csv",
    index=False
)

resumo_final.to_csv(
    pasta / "model_comparison_summary.csv"
)

print("Resultados salvos com sucesso!")
print("Arquivos:", sorted(p.name for p in pasta.iterdir()))

Resultados salvos com sucesso!
Arquivos: ['model_comparison_summary.csv', 'temporal_validation.csv']
